# Step 1 - Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Step 2 - Load clean price files

In [2]:
# Hourly prices — Jan 2020 to Sep 2025
df_h = pd.read_csv("../data/clean/day_ahead_hourly_clean.csv",parse_dates=["timestamp"])

# Quarter-hourly prices — Oct 2025 to Feb 2026
df_qh = pd.read_csv("../data/clean/day_ahead_quarterhourly_clean.csv", parse_dates=["timestamp"])

print("Hourly shape:        ", df_h.shape)
print("Quarter-hourly shape:", df_qh.shape)
print("Hourly range:        ", df_h["timestamp"].min(), "→", df_h["timestamp"].max())
print("QH range:            ", df_qh["timestamp"].min(), "→", df_qh["timestamp"].max())

Hourly shape:         (50394, 2)
Quarter-hourly shape: (14496, 2)
Hourly range:         2020-01-01 00:00:00 → 2025-09-30 23:00:00
QH range:             2025-10-01 00:00:00 → 2026-02-28 23:45:00


# Step 3 - Load exogenous files

In [3]:
# generation forecast (wind + solar)
gen = pd.read_excel( "../data/raw/generation_forecast.xlsx",skiprows=9, header=0,usecols=[0, 4, 5, 6], 
                     names=["timestamp", "wind_offshore_mwh", "wind_onshore_mwh", "solar_mwh"])

# consumption forecast (load)
con = pd.read_excel("../data/raw/consumption_forecast.xlsx", skiprows=9, header=0, usecols=[0, 2], names=["timestamp", "load_mwh"])

# covert the raw string  into a real Python datetime object
for df in [gen, con]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d.%m.%Y %H:%M")

# Convert to numeric if a value cannot to coverted replace it with NaN
for col in ["wind_offshore_mwh", "wind_onshore_mwh", "solar_mwh"]:
    gen[col] = pd.to_numeric(gen[col], errors="coerce")
con["load_mwh"] = pd.to_numeric(con["load_mwh"], errors="coerce")

# Combine wind offshore + onshore into total wind
gen["wind_mwh"] = gen["wind_offshore_mwh"] + gen["wind_onshore_mwh"]
gen = gen[["timestamp", "wind_mwh", "solar_mwh"]]

# Trim both to Oct 2025 – Feb 2026
gen = gen[gen["timestamp"] <= "2026-02-28 23:45"].reset_index(drop=True)
con = con[con["timestamp"] <= "2026-02-28 23:45"].reset_index(drop=True)

print("Generation shape:", gen.shape, "| missing:", gen.isna().sum().sum())
print("Consumption shape:", con.shape, "| missing:", con.isna().sum().sum())

C:\Users\ebteh\miniconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\ebteh\miniconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Generation shape: (14500, 3) | missing: 0
Consumption shape: (14500, 2) | missing: 0


# Step 4 - Build lag-1-day features (QH)

In [4]:
# Lag 1 day = shift by 96 intervals (96 x 15min = 24 hours)
# shift(96) moves the price column 96 rows down, so each row gets the price from the same time yesterday

df_qh["price_lag1d"] = df_qh["price_eur_mwh"].shift(96)

print("Lag-1-day created")

#print how many NaN values
print("Missing values after lag:", df_qh["price_lag1d"].isna().sum(),f"({df_qh['price_lag1d'].isna().sum() / 96:.0f} days)")

Lag-1-day created
Missing values after lag: 96 (1 days)


# Step 5 - Build lag-7-day features (QH)

In [5]:
# Lag 7 days = shift by 672 intervals (96 x 7)
df_qh["price_lag7d"] = df_qh["price_eur_mwh"].shift(672)

print("Lag-7-day created")
print("Missing values after lag:", df_qh["price_lag7d"].isna().sum(),f"({df_qh['price_lag7d'].isna().sum() / 96:.0f} days)")

Lag-7-day created
Missing values after lag: 672 (7 days)


# Step 6 - Build cross-granularity features (hourly to QH)

In [6]:
# For each QH timestamp, find the corresponding hourly price
# from the previous day (floor to hour, shift back 1 day)

# Step 6a: Create a lookup from hourly data
# Turn the hourly dataframe into a dictionary-like object where the timestamp is the key and the price is the value
# Key = timestamp floored to hour, Value = hourly price
df_h_indexed = df_h.set_index("timestamp")["price_eur_mwh"]

# Step 6b: For each QH row, look up the hourly price
# from the same hour on the previous day
def get_hourly_lag(qh_timestamp):
    # Go back 364 days (52 weeks) — lands in the hourly history
    prev_day_hour = (qh_timestamp - pd.Timedelta(days=364)).floor("h")
    return df_h_indexed.get(prev_day_hour, np.nan)

df_qh["price_hourly_lag1d"] = df_qh["timestamp"].apply(get_hourly_lag)

# Also add hourly lag 7 days
def get_hourly_lag7(qh_timestamp):
    # Go back 364 + 7 = 371 days
    prev_week_hour = (qh_timestamp - pd.Timedelta(days=371)).floor("h")
    return df_h_indexed.get(prev_week_hour, np.nan)

df_qh["price_hourly_lag7d"] = df_qh["timestamp"].apply(get_hourly_lag7)

print("Cross-granularity features created")
print("Missing hourly_lag1d:", df_qh["price_hourly_lag1d"].isna().sum())
print("Missing hourly_lag7d:", df_qh["price_hourly_lag7d"].isna().sum())

Cross-granularity features created
Missing hourly_lag1d: 0
Missing hourly_lag7d: 0


# Step 7 - Merge exogenous variables

In [7]:
# Merge generation forecast (wind + solar)
df_qh = df_qh.merge(gen, on="timestamp", how="left")

# Merge consumption forecast (load)
df_qh = df_qh.merge(con, on="timestamp", how="left")

print("After merging exogenous variables:")
print("Shape:", df_qh.shape)
print("Columns:", df_qh.columns.tolist())
print("Missing values per column:")
print(df_qh.isna().sum())

After merging exogenous variables:
Shape: (14508, 9)
Columns: ['timestamp', 'price_eur_mwh', 'price_lag1d', 'price_lag7d', 'price_hourly_lag1d', 'price_hourly_lag7d', 'wind_mwh', 'solar_mwh', 'load_mwh']
Missing values per column:
timestamp               0
price_eur_mwh           0
price_lag1d            96
price_lag7d           672
price_hourly_lag1d      0
price_hourly_lag7d      0
wind_mwh                0
solar_mwh               0
load_mwh                0
dtype: int64


# Step 8 - Drop rows with missing values ( first 7 days)

In [8]:
before = len(df_qh)                                   # count rows BEFORE dropping
df_qh = df_qh.dropna().reset_index(drop=True)          # drop rows with NaN
after  = len(df_qh)                                   # count rows AFTER dropping

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Dropped:     {before - after} rows ({(before - after) / 96:.0f} days)")
print(f"Remaining:   {after / 96:.0f} complete days")
print(f"Date range:  {df_qh['timestamp'].min()} → {df_qh['timestamp'].max()}")

Rows before: 14,508
Rows after:  13,836
Dropped:     672 rows (7 days)
Remaining:   144 complete days
Date range:  2025-10-08 00:00:00 → 2026-02-28 23:45:00


# Step 9 - Check final feature matrix

In [9]:
print("=== FINAL FEATURE MATRIX ===")
print(f"Rows (observations) : {len(df_qh):,}")
print(f"Columns (total)     : {df_qh.shape[1]}")
print(f"  — Target variable : 1  (price_eur_mwh)")
print(f"  — Lag features    : 4  (lag1d, lag7d, hourly_lag1d, hourly_lag7d)")
print(f"  — Exogenous       : 3  (wind, solar, load)")
print()
print(df_qh.describe().T.to_string())

=== FINAL FEATURE MATRIX ===
Rows (observations) : 13,836
Columns (total)     : 9
  — Target variable : 1  (price_eur_mwh)
  — Lag features    : 4  (lag1d, lag7d, hourly_lag1d, hourly_lag7d)
  — Exogenous       : 3  (wind, solar, load)

                      count                        mean                  min                  25%                  50%                  75%                  max          std
timestamp             13836  2025-12-18 22:45:11.318300  2025-10-08 00:00:00  2025-11-12 21:41:15  2025-12-18 22:22:30  2026-01-23 23:03:45  2026-02-28 23:45:00          NaN
price_eur_mwh       13836.0                   98.384258                -1.05                 82.0                95.29               112.75               508.38    40.730652
price_lag1d         13836.0                     98.9693                -1.05              82.3475               95.585             113.0325               508.38    40.879058
price_lag7d         13836.0                   98.356211            

# Step 10 - Save feature matrix 

In [10]:
df_qh.to_csv("../data/features/feature_matrix.csv", index=False)

print("Saved: feature_matrix.csv")
print(f"  Rows    : {len(df_qh):,}")
print(f"  Columns : {df_qh.shape[1]}")
print(f"  Range   : {df_qh['timestamp'].min()} → {df_qh['timestamp'].max()}")

Saved: feature_matrix.csv
  Rows    : 13,836
  Columns : 9
  Range   : 2025-10-08 00:00:00 → 2026-02-28 23:45:00
